https://www.google.com/search?q=duckdb+read+all+excel+sheets&newwindow=1&sca_esv=494ff4588f26de6a&sxsrf=AKL5yDgKZT_PIgKrRl4ITglO8V1Oz3MMBw%3A1793753067652&ei=oX_qaoDxOp2vseMPk9avmQ8&biw=1229&bih=609&uact=5&sclient=gws-wiz-serp&fbs=ACAhTvyChJCcJRW6CZK_ISUtpNyAl1JR-Khf7oJK2HHSzNOXbsffdEXON9bARWkl_VRwhnxobI96KC9pDMoXKQX0h3WlPV8V3jCSa5ubaHw-CiDFOV5bmo8z67QTx4gEiXa4rr4w_K0dxEpTzlW9kv3x2GE1mWHH-yfihwOfZF-fsgLHP4cRnEHF4oQepZgGu5-p43qN3nebpCf2SNICvWByf4kT2LPr_cN--rXPSiI8HLEGkP-X-jW-0O6CRdqGT2jUWE2LH2uE&aep=10&ntc=1&mstk=AUtExfC9TMi_y8QU9UlXuj3jY_DVgOjxJvhaAeQhidcWGICqQG-RvtZ507vHbpwLaNx3_NRYZgCJjkyVL5XbRPRPfL6tOtwzYdPpCNBsL-2K58u5kyoKKa3kaTSBL04HeqrXO8V36Xpp8WgdrBvCALtyvwa9g6MQvIFn63tLkfxwfTZ8oj4QxwXy5116XKr3KGH4qyG-ADi2FoQ-_XS2yGY739zIEoXSK-AtxWrLK8WxCKof1aiXnMZuIDuOC4zItloJ2D-kE6jl-x0TI_6DGwFveTq5nPH_WXV3_t5OOqs3VMsL0NfFSBi1VFDU2U-3726buT95U24DFrKHbg&aioh=3&csuir=1&cs=0&mtid=63_qapTNOdaPseMPoNC08QY&zx=1793753766591&udm=50

In [3]:
import os
from pathlib import Path
import duckdb
import polars as pl
import openpyxl

def get_sqlite_type(pl_type) -> str:
    """Ánh xạ kiểu dữ liệu từ Polars sang SQLite Storage Classes"""
    if pl_type in (pl.Int8, pl.Int16, pl.Int32, pl.Int64, 
                   pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64, 
                   pl.Boolean):
        return "INTEGER"
    elif pl_type in (pl.Float32, pl.Float64):
        return "REAL"
    elif isinstance(pl_type, pl.Decimal):
        return "NUMERIC"
    elif pl_type in (pl.String, pl.Categorical, pl.Date, pl.Datetime, pl.Time):
        return "TEXT"
    elif pl_type == pl.Binary:
        return "BLOB"
    else:
        return "TEXT" # Mặc định cho các kiểu phức hợp (List, Struct)

def process_excel_to_sqlite(excel_path: str, sqlite_path: str):
    # 1. Đảm bảo thư mục chứa SQLite tồn tại
    Path(sqlite_path).parent.mkdir(parents=True, exist_ok=True)
    
    # 2. Khởi tạo DuckDB và nạp các extension
    con = duckdb.connect()
    con.execute("INSTALL excel; LOAD excel;")
    con.execute("INSTALL sqlite; LOAD sqlite;")
    
    # Tắt cấu hình all_varchar để DuckDB không ép mọi thứ thành chuỗi khi ghi
    con.execute("SET sqlite_all_varchar=false;")
    
    # Kết nối an toàn tới file SQLite (Tự động tạo file nếu chưa có)
    con.execute(f"ATTACH IF NOT EXISTS '{sqlite_path}' AS sqlite_db (TYPE SQLITE);")
    
    # 3. Lấy danh sách tất cả các sheet trong file Excel bằng openpyxl
    wb = openpyxl.load_workbook(excel_path, read_only=True)
    sheet_names = wb.sheetnames
    
    print(f"🔄 Đang xử lý file: {os.path.basename(excel_path)}")
    
    for sheet in sheet_names:
        # Tên bảng trong SQLite tương ứng với tên Sheet (xóa khoảng trắng)
        table_name = sheet.replace(" ", "_")
        
        # 4. Dùng Polars đọc quét nhanh dữ liệu để tự động nhận diện Schema chuẩn
        # Thêm các chuỗi lỗi phổ biến vào giá trị rỗng (null_values) để Polars tự nhận diện đúng kiểu SỐ
        df_schema = pl.read_excel(
            source=excel_path,
            sheet_name=sheet,
        ).schema
        
        # 5. Xây dựng câu lệnh CREATE TABLE với kiểu dữ liệu tường minh cho SQLite
        column_defs = []
        for col_name, pl_type in df_schema.items():
            sqlite_type = get_sqlite_type(pl_type)
            column_defs.append(f'"{col_name}" {sqlite_type}')
            
        create_table_sql = f"""
            CREATE TABLE IF NOT EXISTS sqlite_db.{table_name} (
                {", ".join(column_defs)}
            );
        """
        
        # 6. Thực thi tạo bảng trống chuẩn kiểu dữ liệu trong SQLite (Nếu đã có bảng từ trước thì bỏ qua)
        con.execute(create_table_sql)
        
        # 7. Dùng DuckDB đọc trực tiếp từ Excel, biến chuỗi 'NULL' thành ô trống thật, và nạp vào SQLite
        # Sử dụng INSERT OR IGNORE hoặc INSERT tuỳ nhu cầu (dưới đây dùng INSERT để nạp tiếp dữ liệu)
        # Nếu muốn xóa dữ liệu cũ của bảng trước khi nạp lại, bỏ comment dòng dưới:
        # con.execute(f"DELETE FROM sqlite_db.{table_name};")
        
        insert_sql = f"""
            INSERT INTO sqlite_db.{table_name}
            SELECT * FROM read_xlsx(?, sheet=?);
        """
        con.execute(insert_sql, [excel_path, sheet])
        print(f"   └── Đã nạp xong sheet '{sheet}' vào bảng sqlite_db.{table_name}")
        
    # 8. Kết thúc quy trình, ngắt kết nối an toàn
    con.execute("DETACH sqlite_db;")
    con.close()
    print(f"🎉 Hoàn thành xử lý file Excel vào {sqlite_path}\n")

# --- VÍ DỤ SỬ DỤNG HÀNG LOẠT FILE EXCEL ---
if __name__ == "__main__":
    # Đường dẫn file SQLite của bạn
    TARGET_SQLITE = "my_final_database.db"
    
    # Danh sách các file Excel cần xử lý
    excel_files = [
        "/home/user/py-pannguyen-learn/data/raw/sales_data_sample.xlsx"
    ]
    
    # Hoặc quét toàn bộ file Excel trong 1 thư mục:
    # import glob
    # excel_files = glob.glob("D:/data/source/*.xlsx")
    
    for file in excel_files:
        if os.path.exists(file):
            process_excel_to_sqlite(file, TARGET_SQLITE)


🔄 Đang xử lý file: sales_data_sample.xlsx


ConversionException: Conversion Error: Could not convert string 'Quantity' to INT64 when casting from source column G1